In [ ]:
import os
import pickle
import jsonlines
import pandas as pd
import numpy as np
import json
import copy
from tqdm import tqdm


In [ ]:
data = json.load(open("./handled/item2attributes.json", "r"))

In [ ]:
instruction = "The beauty item has the following attributes: \n "
image_instruction = "The beauty product has the image: "

In [ ]:
item_data = {}
for item_dict in tqdm(data.values()):
    item_prompt = copy.deepcopy(instruction)
    item_id = None
    for key, value in item_dict.items():
        if key in ["related", "imUrl", "salesRank"]:   # drop longitude and latitude
            continue
        elif key in ["asin"]:  # get the item id
            item_id = value
        elif key in ["categories"]:    # list type attributes
            attri_str = ""
            for meta_str in value[0]:
                attri_str += (meta_str + ", ")
            if len(value) == 0:
                attri_str = "none, "
            attri_str = attri_str.replace("\n", " ").replace(";", ".")
            if len(attri_str) > 100:
                attri_str = attri_str[:100]
            attri_prompt = key + " is " + attri_str[:-2] + "; "    # [:-2] is to remove the last ", "
            item_prompt += attri_prompt
        else:   # str type attributes 
            if len(str(value)) > 100:
                value = value[:100]
            attri_prompt = key + " is " + str(value).replace("\n", " ").replace(";", ".") + "; "
            item_prompt += attri_prompt
    if item_id:
        item_data[item_id] = item_prompt[:-2]
    else:
        raise ValueError("No item id")

In [ ]:
id2Image_path = {}
for path, dir_list, file_list in os.walk(f"./handled/image/"):
    for file in file_list:
        id = file.split('.')[0]
        id2Image_path[id] = f"{file}.jpg"
print(len(id2Image_path))

In [ ]:
# convert to jsonline
def save_data(data_path, data):
    '''write all_data list to a new jsonl'''
    with jsonlines.open("./handled/"+ data_path, "w") as w:
        for meta_data in data:
            w.write(meta_data)

id_map = json.load(open("./handled/id_map.json", "r"))["item2id"]
json_data = []
for key, value in item_data.items():
    json_data.append({"attributes_input": value, "image_instruction_input": image_instruction, "image_url": id2Image_path.get(key, ""), "item": key, "item_id": int(id_map[key])})
# 按照 item_id 进行递增排序
json_data.sort(key=lambda x: x["item_id"])
save_data("multimodal_item_str.jsonline", json_data)

./handled/image/
B00006IV30.jpg


ValueError: 

In [ ]:
# !jupyter nbconvert --to script prepare_data.ipynb

In [ ]:
# import os
# import tqdm
# def rename_files(directory):
#     for root, _, files in os.walk(directory):
#         for file in files:
#             if file.startswith("._"):
#                 print(file)
#                 # old_path = os.path.join(root, file)
#                 # new_path = os.path.join(root, file[2:])  # 去掉 "._" 前缀
#                 # os.rename(old_path, new_path)
#                 # # print(f'Renamed: {old_path} -> {new_path}')

# # 指定要遍历的文件夹路径
# folder_path = "./raw/image"  # 修改为你的文件夹路径
# rename_files(folder_path)